## 기본 예시: 프롬프트 + 모델 + 출력 파서

가장 기본적이고 일반적인 사용 사례는 prompt 템플릿과 모델을 함께 연결하는 것입니다. 이것이 어떻게 작동하는지 보기 위해, 각 나라별 수도를 물어보는 Chain을 생성해 보겠습니다.


In [1]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

True

In [2]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
# !pip install -qU langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH01-Basic")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH01-Basic


## 프롬프트 템플릿의 활용

`PromptTemplate`

- 사용자의 입력 변수를 사용하여 완전한 프롬프트 문자열을 만드는 데 사용되는 템플릿입니다
- 사용법
  - `template`: 템플릿 문자열입니다. 이 문자열 내에서 중괄호 `{}`는 변수를 나타냅니다.
  - `input_variables`: 중괄호 안에 들어갈 변수의 이름을 리스트로 정의합니다.

`input_variables`

- input_variables는 PromptTemplate에서 사용되는 변수의 이름을 정의하는 리스트입니다.

In [3]:
from langchain_teddynote.messages import stream_response  # 스트리밍 출력
from langchain_core.prompts import PromptTemplate

`from_template()` 메소드를 사용하여 PromptTemplate 객체 생성


In [4]:
# template 정의
template = "{country}의 수도는 어디인가요?"

# from_template 메소드를 이용하여 PromptTemplate 객체 생성
prompt_template = PromptTemplate.from_template(template)
prompt_template

PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디인가요?')

In [5]:
# prompt 생성
prompt = prompt_template.format(country="대한민국")
prompt

'대한민국의 수도는 어디인가요?'

In [6]:
# prompt 생성
prompt = prompt_template.format(country="미국")
prompt

'미국의 수도는 어디인가요?'

In [7]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-3.5-turbo",
    max_tokens=2048,
    temperature=0.1,
)

## Chain 생성

### LCEL(LangChain Expression Language)

![lcel.png](./images/lcel.png)

여기서 우리는 LCEL을 사용하여 다양한 구성 요소를 단일 체인으로 결합합니다

```
chain = prompt | model | output_parser
```

`|` 기호는 [unix 파이프 연산자](<https://en.wikipedia.org/wiki/Pipeline_(Unix)>)와 유사하며, 서로 다른 구성 요소를 연결하고 한 구성 요소의 출력을 다음 구성 요소의 입력으로 전달합니다.

이 체인에서 사용자 입력은 프롬프트 템플릿으로 전달되고, 그런 다음 프롬프트 템플릿 출력은 모델로 전달됩니다. 각 구성 요소를 개별적으로 살펴보면 무슨 일이 일어나고 있는지 이해할 수 있습니다.


In [8]:
# prompt 를 PromptTemplate 객체로 생성합니다.
prompt = PromptTemplate.from_template("{topic} 에 대해 쉽게 설명해주세요.")

model = ChatOpenAI()

chain = prompt | model

### invoke() 호출

- python 딕셔너리 형태로 입력값을 전달합니다.(키: 값)
- invoke() 함수 호출 시, 입력값을 전달합니다.

In [9]:
# input 딕셔너리에 주제를 '인공지능 모델의 학습 원리'으로 설정합니다.
input = {"topic": "인공지능 모델의 학습 원리"}

In [10]:
# prompt 객체와 model 객체를 파이프(|) 연산자로 연결하고 invoke 메서드를 사용하여 input을 전달합니다.
# 이를 통해 AI 모델이 생성한 메시지를 반환합니다.
chain.invoke(input)

AIMessage(content='인공지능 모델의 학습 원리는 데이터를 입력받고 이를 처리하여 원하는 결과를 출력하는 과정입니다. 이때 모델은 입력 데이터와 출력 결과 사이의 패턴이나 관계를 학습하고 이를 기반으로 새로운 데이터에 대해 예측이나 분류를 수행합니다.\n\n모델은 초기에는 무작위로 설정된 가중치와 편향을 가지고 있어 입력 데이터를 처리하는 과정에서 이를 조절하며 학습을 진행합니다. 학습은 주어진 데이터를 이용하여 손실 함수를 최소화하는 방향으로 가중치와 편향을 조정하면서 이루어집니다. 손실 함수는 모델의 출력 결과와 실제 결과 사이의 차이를 나타내며, 이 차이가 작을수록 모델이 더욱 정확하게 학습하고 있는 것을 의미합니다.\n\n이렇게 학습된 모델은 새로운 데이터를 입력받아 예측이나 분류를 수행할 수 있습니다. 따라서 인공지능 모델의 학습 원리는 데이터를 기반으로 가중치와 편향을 조정하여 원하는 결과를 출력하는 것에 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 367, 'prompt_tokens': 33, 'total_tokens': 400, 'completion_tokens_details': {'audio_tokens': None, 'reasoning_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-5c211c48-5e38-4c77-8b59-f3a842b81fd6-0', usage_metadata={'input_tokens': 33, 'output_tokens': 367, 'total_tokens': 400, 

아래는 스트리밍을 출력하는 예시 입니다.

In [11]:
# 스트리밍 출력을 위한 요청
answer = chain.stream(input)
# 스트리밍 출력
stream_response(answer)

인공지능 모델의 학습 원리는 데이터를 입력으로 받아들이고, 이 데이터를 분석하여 패턴을 찾아내는 과정입니다. 모델은 이러한 데이터와 패턴을 바탕으로 스스로 학습하고, 새로운 데이터에 대한 예측을 수행할 수 있습니다.

이러한 학습 과정은 주로 두 가지 방법으로 이루어집니다. 첫 번째는 지도 학습이라고 하는 방법으로, 모델에게 입력 데이터와 정답 레이블을 함께 제공하여 학습시키는 방식입니다. 모델은 이러한 데이터를 통해 입력과 출력 간의 관계를 학습하고, 새로운 데이터에 대한 예측을 수행할 수 있습니다.

두 번째는 비지도 학습이라고 하는 방법으로, 모델에게 정답 레이블 없이 데이터만을 제공하여 학습시키는 방식입니다. 모델은 이러한 데이터를 통해 숨겨진 패턴이나 구조를 찾아내고, 이를 기반으로 새로운 데이터에 대한 예측을 수행할 수 있습니다.

이러한 과정을 반복하면 모델은 점차적으로 더 정확한 예측을 수행할 수 있게 되며, 이를 통해 다양한 분야에서 인공지능 기술을 활용할 수 있습니다.

### 출력파서(Output Parser)


In [12]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

Chain 에 출력파서를 추가합니다.

In [13]:
# 프롬프트, 모델, 출력 파서를 연결하여 처리 체인을 구성합니다.
chain = prompt | model | output_parser

In [14]:
# chain 객체의 invoke 메서드를 사용하여 input을 전달합니다.
input = {"topic": "인공지능 모델의 학습 원리"}
chain.invoke(input)

'인공지능 모델의 학습 원리는 데이터를 입력으로 받아서 내부적으로 학습 알고리즘을 통해 패턴을 학습하는 과정입니다. 모델은 입력 데이터를 분석하여 입력과 출력 간의 관계를 파악하고, 이 관계를 나타내는 함수를 찾아내는 것이 목표입니다.\n\n모델은 초기에는 무작위로 설정된 가중치와 편향을 가지고 있습니다. 이후 입력 데이터를 사용하여 출력을 예측하고, 이 예측값과 실제값의 차이를 최소화하는 방향으로 가중치와 편향을 조정하면서 학습을 진행합니다. 이 과정을 반복하면서 모델은 데이터 간의 패턴을 스스로 학습하게 되고, 최종적으로 원하는 목표를 달성하는데 도움이 되는 함수를 학습하게 됩니다. \n\n이러한 방식으로 모델은 학습 데이터에서 학습된 패턴을 새로운 데이터에 적용하여 적절한 출력을 생성할 수 있게 됩니다. 이러한 학습 원리를 통해 인공지능 모델은 다양한 작업을 수행하고 문제를 해결할 수 있습니다.'

In [15]:
# 스트리밍 출력을 위한 요청
answer = chain.stream(input)
# 스트리밍 출력
stream_response(answer)

인공지능 모델의 학습 원리는 데이터를 사용하여 패턴이나 규칙을 학습하는 것입니다. 모델은 입력 데이터를 받아들이고 일련의 수학적 연산을 통해 이 데이터에 내재된 패턴을 발견하려고 합니다. 이 과정에서 모델은 손실 함수를 사용하여 예측과 실제 값 사이의 차이를 계산하고 이를 최소화하려고 노력합니다.

모델이 학습을 통해 최적의 예측을 하기 위해서는 대량의 데이터가 필요합니다. 더 많은 데이터를 이용하면 모델은 더 정확한 패턴을 학습할 수 있습니다. 또한, 모델의 구조와 하이퍼파라미터를 조정하여 학습 과정을 최적화할 수 있습니다.

최종적으로, 학습된 모델은 새로운 데이터에 대해 예측을 수행할 수 있게 되며, 이를 통해 다양한 분야에서 유용하게 활용될 수 있습니다.

### 템플릿을 변경하여 적용

- 아래의 프롬프트 내용을 얼마든지 **변경** 하여 테스트 해볼 수 있습니다.
- `model_name` 역시 변경하여 테스트가 가능합니다.

In [16]:
template = """
당신은 영어를 가르치는 10년차 영어 선생님입니다. 주어진 상황에 맞는 영어 회화를 작성해 주세요.
양식은 [FORMAT]을 참고하여 작성해 주세요.

#상황:
{question}

#FORMAT:
- 영어 회화:
- 한글 해석:
"""

# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOpenAI 챗모델을 초기화합니다.
model = ChatOpenAI(model_name="gpt-4-turbo")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

In [17]:
# 체인을 구성합니다.
chain = prompt | model | output_parser

In [18]:
# 완성된 Chain을 실행하여 답변을 얻습니다.
print(chain.invoke({"question": "저는 식당에 가서 음식을 주문하고 싶어요"}))

- 영어 회화:
  1. (Customer) Hi, I’d like to have a table for two, please.
  2. (Waiter) Right this way. Here’s the menu. What would you like to order?
  3. (Customer) Can I have the grilled salmon and a glass of white wine, please?
  4. (Waiter) Of course. Would you like any sides or a salad with that?
  5. (Customer) Yes, a Caesar salad and some garlic bread, please.
  6. (Waiter) Great choice! I'll have your order right up.

- 한글 해석:
  1. (손님) 안녕하세요, 두 명이서 식사할 수 있는 테이블로 부탁드려요.
  2. (웨이터) 이쪽으로 오세요. 여기 메뉴판입니다. 무엇을 주문하시겠어요?
  3. (손님) 구운 연어와 화이트 와인 한 잔 주문할 수 있을까요?
  4. (웨이터) 물론입니다. 사이드 메뉴나 샐러드도 추가하시겠어요?
  5. (손님) 네, 시저 샐러드와 마늘빵 주세요.
  6. (웨이터) 좋은 선택이세요! 주문 바로 준비하겠습니다.


In [19]:
# 완성된 Chain을 실행하여 답변을 얻습니다.
# 스트리밍 출력을 위한 요청
answer = chain.stream({"question": "저는 식당에 가서 음식을 주문하고 싶어요"})
# 스트리밍 출력
stream_response(answer)

- 영어 회화:
  - Server: Hello, welcome to our restaurant! How many are dining tonight?
  - You: Just me, thank you.
  - Server: Great, please follow me to your table. Here's the menu. Are you ready to order or need a few minutes?
  - You: I think I'm ready to order. Could I get the grilled salmon with a side of vegetables, please?
  - Server: Of course! Would you like anything to drink?
  - You: Yes, could I have a glass of white wine?
  - Server: Sure, I'll bring that right out with your meal.

- 한글 해석:
  - 종업원: 안녕하세요, 저희 식당에 오신 것을 환영합니다! 몇 분이 드실 예정인가요?
  - 당신: 저 혼자입니다, 감사합니다.
  - 종업원: 좋습니다, 이쪽으로 자리로 안내해 드리겠습니다. 여기 메뉴판입니다. 주문하실 준비가 되셨나요, 아니면 조금 더 시간이 필요하신가요?
  - 당신: 주문할 준비가 된 것 같아요. 구운 연어와 채소 사이드를 주문할 수 있을까요?
  - 종업원: 물론입니다! 음료는 무엇으로 드릴까요?
  - 당신: 네, 화이트 와인 한 잔 주시겠어요?
  - 종업원: 알겠습니다, 식사와 함께 곧 가져다 드리겠습니다.

In [20]:
# 이번에는 question 을 '미국에서 피자 주문'으로 설정하여 실행합니다.
# 스트리밍 출력을 위한 요청
answer = chain.stream({"question": "미국에서 피자 주문"})
# 스트리밍 출력
stream_response(answer)

- 영어 회화:
  Customer: Hi, I'd like to place an order for delivery.
  Employee: Sure, what would you like to order?
  Customer: I'll have a large pepperoni pizza with extra cheese, please.
  Employee: Great choice! Anything to drink with that?
  Customer: Yes, two cans of Coke, please.
  Employee: Alright, that’s a large pepperoni pizza with extra cheese and two cans of Coke. Will that be everything?
  Customer: Yes, that's all. How long will the delivery take?
  Employee: It should take about 30-45 minutes. Can I have your address, please?
  Customer: It's 742 Evergreen Terrace.
  Employee: Thank you. Your total comes to $24.50. Our driver will see you in about 30-45 minutes.
  Customer: Perfect, thanks!

- 한글 해석:
  고객: 안녕하세요, 배달 주문하고 싶습니다.
  직원: 네, 무엇을 주문하시겠어요?
  고객: 큰 사이즈 페퍼로니 피자에 치즈 추가로 주세요.
  직원: 좋은 선택이네요! 음료는 필요하신가요?
  고객: 네, 콜라 두 캔 주세요.
  직원: 알겠습니다. 큰 사이즈 페퍼로니 피자에 치즈 추가와 콜라 두 캔이요. 다른 주문은 없으신가요?
  고객: 네, 그게 다입니다. 배달은 얼마나 걸리나요?
  직원: 대략 30-45분 정도 걸립니다. 주소를 알려주시겠어요?
  고객: 742 에버그린 테라